# FEST evaluation

In [ ]:
# Clone the repository (skip if already exists)
import os
if not os.path.exists('synprivutil'):
    !git clone https://github.com/Karo2222/synprivutil.git

# Move into the directory so you can see the files
%cd synprivutil

c:\Users\imets\UZH\Masters_project\synthetic_network_data_gen\FEST_eval\synprivutil


C:\Users\imets\AppData\Roaming\Python\Python310\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Prerequisites
The python version 3.10 was used to develop this framework. For the following packages, these versions were used:

Numpy version: 1.26.4
Pandas version: 2.2.2
SDV version: 1.15.0
Scikit-learn version: 1.5.1
Seaborn version: 0.12.2
Matplotlib version: 3.9.2
RDT version: 1.12.3
Anonymeter version: 1.0.0
Scipy version: 1.13.0
Dython version: 0.7.8
OT version: 0.9.4

In [1]:
%cd synprivutil

c:\Users\imets\UZH\Masters_project\synthetic_network_data_gen\FEST_eval\synprivutil


C:\Users\imets\AppData\Roaming\Python\Python310\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import sys
import os
import pandas as pd
from sdv.metadata import SingleTableMetadata

# Add the synprivutil directory to the path
synprivutil_path = os.path.join(os.getcwd(), 'synprivutil')
if synprivutil_path not in sys.path:
    sys.path.append(synprivutil_path)

from privacy_utility_framework.privacy_utility_framework.synthesizers.synthesizers import GaussianMixtureModel

# Load original dataset
original_data = pd.read_csv('examples/insurance_datasets/train/insurance.csv')

# Create metadata for the dataset
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(original_data)

# Initialize the Gaussian Mixture Model with a max of 10 components
gmm_model = GaussianMixtureModel(max_components=10)

# Fit the model on the original data
gmm_model.fit(original_data)

# Generate synthetic data
synthetic_data = gmm_model.sample(len(original_data))

# Save synthetic data to a CSV file
gmm_model.save_sample("gmm_sample.csv", len(original_data))

print("Synthetic data generated and saved to gmm_sample.csv.")
print(synthetic_data.head())

Data saved to gmm_sample.csv
Synthetic data generated and saved to gmm_sample.csv.
   age     sex        bmi  children smoker     region       charges
0   12  female  43.167515         0     no  southeast  13202.831359
1   42  female  40.472262         1     no  southeast  24330.630679
2   11    male  40.696603         1     no  southeast  13671.221098
3   14  female  30.153420         0    yes  southeast  11764.266335
4   38    male  40.071285         0     no  southwest  23407.846723


In [3]:
import pandas as pd
import sys
import os

# Add the synprivutil directory to the path
synprivutil_path = os.path.join(os.getcwd(), 'synprivutil')
if synprivutil_path not in sys.path:
    sys.path.append(synprivutil_path)

from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.privacy_metrics.privacy_metric_manager import PrivacyMetricManager
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.privacy_metrics.distance.adversarial_accuracy_class import AdversarialAccuracyCalculator
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.privacy_metrics.distance.dcr_class import DCRCalculator
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.privacy_metrics.distance.nndr_class import NNDRCalculator

# Load data
original_data = pd.read_csv('examples/insurance_datasets/train/insurance.csv')
synthetic_data = pd.read_csv('gmm_sample.csv')

# Encode categorical columns identically for both datasets
original_data['source'] = 'real'
synthetic_data['source'] = 'synthetic'
combined = pd.concat([original_data, synthetic_data], axis=0)

categorical_cols = combined.select_dtypes(include=['object']).columns.tolist()
categorical_cols.remove('source')

combined_encoded = pd.get_dummies(combined, columns=categorical_cols, dtype=int)

original_data = combined_encoded[combined_encoded['source'] == 'real'].drop('source', axis=1)
synthetic_data = combined_encoded[combined_encoded['source'] == 'synthetic'].drop('source', axis=1)

original_name = "Insurance"
synthetic_name = "GMM_Synthetic"

# Create privacy metric manager
privman = PrivacyMetricManager()

# Add metrics
metric_list = [
    DCRCalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name),
    NNDRCalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name),
    AdversarialAccuracyCalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name)
]

privman.add_metric(metric_list)

# Evaluate and print results
results = privman.evaluate_all()
for key, value in results.items():
    print(f"{key}: {value}")

DCRCalculator('Insurance', 'GMM_Synthetic'): 0.15922857162691628
NNDRCalculator('Insurance', 'GMM_Synthetic'): 0.7585015994091323
AdversarialAccuracyCalculator('Insurance', 'GMM_Synthetic'): 0.5892523364485981


In [4]:
import pandas as pd
import sys

# Add the synprivutil directory to the path
synprivutil_path = os.path.join(os.getcwd(), 'synprivutil')
if synprivutil_path not in sys.path:
    sys.path.append(synprivutil_path)

from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.utility_metrics.utility_metric_manager import UtilityMetricManager
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.utility_metrics.statistical.basic_stats import BasicStatsCalculator
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.utility_metrics.statistical.correlation import CorrelationCalculator
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.utility_metrics.statistical.js_similarity import JSCalculator
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.utility_metrics.statistical.ks_test import KSCalculator
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.utility_metrics.statistical.mutual_information import MICalculator
from synprivutil.privacy_utility_framework.privacy_utility_framework.metrics.utility_metrics.statistical.wasserstein import WassersteinCalculator
# Load data
original_data = pd.read_csv('examples/insurance_datasets/train/insurance.csv')
synthetic_data = pd.read_csv('gmm_sample.csv')

# Encode categorical columns identically for both datasets
original_data['source'] = 'real'
synthetic_data['source'] = 'synthetic'
combined = pd.concat([original_data, synthetic_data], axis=0)

categorical_cols = combined.select_dtypes(include=['object']).columns.tolist()
categorical_cols.remove('source')

combined_encoded = pd.get_dummies(combined, columns=categorical_cols, dtype=int)

original_data = combined_encoded[combined_encoded['source'] == 'real'].drop('source', axis=1)
synthetic_data = combined_encoded[combined_encoded['source'] == 'synthetic'].drop('source', axis=1)

original_name = "Insurance"
synthetic_name = "GMM_Synthetic"

# Create privacy metric manager
utman = UtilityMetricManager()

metric_list = [
    BasicStatsCalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name),
    CorrelationCalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name),
    JSCalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name),
    KSCalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name),
    MICalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name),
    # WassersteinCalculator(original_data, synthetic_data, original_name=original_name, synthetic_name=synthetic_name),
]
utman.add_metric(metric_list)
results = utman.evaluate_all()

for key, value in results.items():
    print(f"{key}: {value}")

Method CorrelationMethod.PEARSON was used.
BasicStatsCalculator('Insurance', 'GMM_Synthetic'): {'mean': 0.009110102617809979, 'median': 0.0023223545535458446, 'var': 0.004361178508024739}
CorrelationCalculator('Insurance', 'GMM_Synthetic'): 0.9864216019995133
JSCalculator('Insurance', 'GMM_Synthetic'): 0.9356230919499157
KSCalculator('Insurance', 'GMM_Synthetic'): 0.9692367601246105
MICalculator('Insurance', 'GMM_Synthetic'): 0.992169122987004
